1. Import the movie review data as a data frame and ensure that the data is loaded properly.

In [1]:
import pandas as pd

#load dataset
df = pd.read_csv('data/labeledTrainData.tsv', sep='\t')

#preview
df.head()

,id,sentiment,review
0,5814_8,1,With all this stuff going down at the moment w...
1,2381_9,1,"\The Classic War of the Worlds\"" by Timothy Hi..."
2,7759_3,0,The film starts with a manager (Nicholas Bell)...
3,3630_4,0,It must be assumed that those who praised this...
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...


2. How many of each positive and negative reviews are there?

In [2]:
#counting the data
df['sentiment'].value_counts()

sentiment
1    12500
0    12500
Name: count, dtype: int64

I was not expecting sentiment to be perfectly evenly split. Let's explore further.

3. Use TextBlob to classify each movie review as positive or negative. Assume that a polarity score greater than or equal to zero is a positive sentiment and less than 0 is a negative sentiment.

In [ ]:
!pip install textblob

In [ ]:
import nltk
nltk.download('punkt')

In [ ]:
!python -m textblob.download_corpora

In [3]:
from textblob import TextBlob

#calculating polarity
def get_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    return 1 if polarity >= 0 else 0

df['tb_sentiment'] = df['review'].apply(get_sentiment)

4. Check the accuracy of this model. Is this model better than random guessing?

In [4]:
from sklearn.metrics import accuracy_score

#checking accuracy of dataframes from above
accuracy = accuracy_score(df['sentiment'], df['tb_sentiment'])
print("TextBlob Accuracy:", accuracy)

TextBlob Accuracy: 0.68524


Based on the score of .68, the accuracy is better than random guessing. Random guessing is considered .5

5. For up to five points extra credit, use another prebuilt text sentiment analyzer, e.g., VADER, and repeat steps (3) and (4).

In [5]:
from nltk.sentiment import SentimentIntensityAnalyzer

#VADER download
nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

#calculating polarity
def vader_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    return 1 if score >= 0 else 0

df['vader_sentiment'] = df['review'].apply(vader_sentiment)

#checking accuracy of dataframes
vader_accuracy = accuracy_score(df['sentiment'], df['vader_sentiment'])
print("VADER Accuracy:", vader_accuracy)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Chris\AppData\Roaming\nltk_data...


VADER Accuracy: 0.69356


VADER accuracy is pretty similar to TextBlob. It is a slight improvement. TextBlob is .685 while VADER is .693. Both are more accurate than random guessing.

Part 2: Prepping Text for a Custom Model

1-4. Convert all text to lowercase. Remove punctuation and special characters. Remove stop words. Apply NLTK's PorterStemmer.

In [6]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    #making text lowercase
    text = text.lower()
    
    #removing punctuation & special chars
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    #tokenizing
    words = text.split()
    
    #removing stopwords
    words = [w for w in words if w not in stop_words]
    
    #stemming
    words = [stemmer.stem(w) for w in words]
    
    return ' '.join(words)

df['clean_review'] = df['review'].apply(preprocess)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Chris\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


5. Create a bag-of-words matrix from your stemmed text (output from (4)), where each row is a word-count vector for a single movie review (see sections 5.3 & 6.8 in the Machine Learning with Python Cookbook). Display the dimensions of your bag-of-words matrix. The number of rows in this matrix should be the same as the number of rows in your original data frame.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(df['clean_review'])

print("BoW shape:", bow_matrix.shape)

BoW shape: (25000, 89468)


The output for rows is 25000 which is the same as the original data frame.

6. Create a term frequency-inverse document frequency (tf-idf) matrix from your stemmed text, for your movie reviews (see section 6.9 in the Machine Learning with Python Cookbook). Display the dimensions of your tf-idf matrix. These dimensions should be the same as your bag-of-words matrix.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['clean_review'])

print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (25000, 89468)


The dimensions are 25000, 89468 which is the same as my bag-of-words matrix.